# ЛР2 по НКС: QLoRA fine-tuning для варианта 20

Ноутбук подготовлен под Google Colab и использует датасет из ЛР1.

Что нужно загрузить в Colab:
- файл `dataset.zip`

Что делает ноутбук:
- проверяет GPU;
- загружает датасет из ЛР1;
- загружает базовую модель в 4-bit;
- сохраняет ответы базовой модели на тематические промпты;
- дообучает модель через QLoRA;
- сохраняет LoRA-адаптер;
- сравнивает ответы до и после дообучения;
- строит график loss и сохраняет результаты в файлы.

In [ ]:
!pip install -q -U transformers datasets accelerate bitsandbytes peft trl sentencepiece wandb

In [ ]:
import gc
import json
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from datasets import load_from_disk
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)

# SFTTrainer используется для supervised fine-tuning поверх causal LM.
from trl import SFTConfig, SFTTrainer

In [ ]:
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1024**3:.2f} GB")
    major, minor = torch.cuda.get_device_capability(0)
    print(f"Compute capability: {major}.{minor}")
else:
    raise RuntimeError("GPU не обнаружен. Для QLoRA в Colab нужен runtime с GPU.")

## Загрузка датасета из ЛР1

В этой ячейке загрузите `dataset.zip` из папки `ЛР1`.

In [ ]:
from google.colab import files

# Загрузка архива с датасетом ЛР1 в текущую Colab-сессию.
uploaded = files.upload()
zip_name = next(iter(uploaded))
print(f"Загружен файл: {zip_name}")

In [ ]:
import shutil
import zipfile

# Очистка старой распаковки и извлечение архива с датасетом.
shutil.rmtree("/content/lab1_data", ignore_errors=True)
os.makedirs("/content/lab1_data", exist_ok=True)

with zipfile.ZipFile(zip_name, "r") as zf:
    zf.extractall("/content/lab1_data")

print("Архив распакован в /content/lab1_data")

In [ ]:
candidate_paths = [
    "/content/lab1_data/corpus_variant_20",
    "/content/lab1_data/dataset/corpus_variant_20",
    "/content/lab1_data/dataset",
]

dataset_path = None
for path in candidate_paths:
    if os.path.isdir(path) and os.path.exists(os.path.join(path, "state.json")):
        dataset_path = path
        break

if dataset_path is None:
    raise FileNotFoundError("Не удалось найти Hugging Face dataset после распаковки dataset.zip")

print(f"Найден путь к датасету: {dataset_path}")

In [ ]:
dataset = load_from_disk(dataset_path)
print(dataset)
print("Колонки:", dataset.column_names)
print("Количество примеров:", len(dataset))
print("Пример текста:\n")
print(dataset[0]["text"][:1200])

In [ ]:
VARIANT = 20
TOPIC = "Специалист по гибридным системам"
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 512
OUTPUT_DIR = Path("/content/variant20_qlora_output")
ADAPTER_DIR = OUTPUT_DIR / "lora_adapter_variant20"
RESULTS_DIR = OUTPUT_DIR / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Вариант: {VARIANT}")
print(f"Тема: {TOPIC}")
print(f"Базовая модель: {MODEL_NAME}")

In [ ]:
dataset = dataset.remove_columns([c for c in dataset.column_names if c != "text"])

# Нормализация текста нужна для выравнивания пробелов перед обучением.
def normalize_example(example):
    text = " ".join(example["text"].split())
    return {"text": text}

normalized_dataset = dataset.map(normalize_example)
# Деление на train/validation для контроля качества на отложенной выборке.
split_dataset = normalized_dataset.train_test_split(test_size=0.15, seed=42)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

print(f"Всего примеров: {len(normalized_dataset)}")
print(f"Train: {len(train_dataset)}")
print(f"Validation: {len(val_dataset)}")
print("Пример train-текста:\n")
print(train_dataset[0]["text"][:500])

In [ ]:
use_bf16 = False
compute_dtype = torch.float16

# 4-битная конфигурация QLoRA для экономии памяти в Colab.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Базовая модель загружается в квантизованном виде.
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
base_model.config.torch_dtype = torch.float16
base_model.config.use_cache = False

print("Модель и токенизатор загружены")
print("compute_dtype:", compute_dtype)
print("model.config.torch_dtype:", getattr(base_model.config, "torch_dtype", None))

In [ ]:
test_prompts = [
    "Гибридные рекомендательные системы объединяют",
    "Проблема cold start возникает, когда",
    "Netflix Prize повлиял на развитие рекомендательных систем тем, что",
    "Коллаборативная фильтрация полезна, потому что",
    "Контентная фильтрация отличается от коллаборативной тем, что",
]

# Генерация используется для сравнения базовой и дообученной моделей.
def generate_text(model, prompts, max_new_tokens=120):
    results = []
    model.eval()
    device = next(model.parameters()).device
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
            )
        text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        results.append({"prompt": prompt, "response": text})
    return results

base_results = generate_text(base_model, test_prompts)
for item in base_results:
    print("=" * 80)
    print("PROMPT:", item["prompt"])
    print("RESPONSE:", item["response"])

with open(RESULTS_DIR / "base_model_outputs.json", "w", encoding="utf-8") as f:
    json.dump(base_results, f, ensure_ascii=False, indent=2)

## Настройка QLoRA

## Дополнительно: WandB

Блок опциональный. Если логирование в WandB не требуется, `USE_WANDB` нужно оставить равным `False`.

In [ ]:
USE_WANDB = False
WANDB_PROJECT = "nks-lab2"
WANDB_RUN_NAME = "variant20-qlora-r16"

if USE_WANDB:
    import wandb
    wandb.login()
    wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        config={
            "variant": VARIANT,
            "topic": TOPIC,
            "model_name": MODEL_NAME,
            "max_seq_length": MAX_SEQ_LENGTH,
        },
    )
    print("WandB logging enabled")
else:
    print("WandB logging disabled")

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Подготовка модели к k-bit обучению и подключение LoRA-адаптеров.
model = prepare_model_for_kbit_training(base_model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Основные параметры обучения и валидации.
training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_steps=5,
    logging_steps=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    fp16=False,
    bf16=False,
    per_device_eval_batch_size=1,
    report_to="wandb" if USE_WANDB else "none",
    run_name=WANDB_RUN_NAME if USE_WANDB else None,
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=False,
)

# Trainer получает train и validation выборки и запускает SFT-обучение.
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    processing_class=tokenizer,
)

In [ ]:
print("Запуск обучения...")
start_time = time.time()
train_result = trainer.train()
training_time_sec = time.time() - start_time
# Итоговая валидация берется из log_history, который заполняется в конце эпох.
eval_history = [item for item in trainer.state.log_history if "eval_loss" in item]
eval_results = eval_history[-1] if eval_history else {}
print("Validation metrics:", eval_results)
if USE_WANDB:
    wandb.log({"final_eval_loss": eval_results.get("eval_loss")})
print(f"Обучение завершено за {training_time_sec / 60:.2f} мин.")

In [ ]:
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"LoRA-адаптер сохранен в: {ADAPTER_DIR}")

# Подсчет итогового размера адаптера нужен для анализа эффективности LoRA.
adapter_size = 0
for root, _, files_in_dir in os.walk(ADAPTER_DIR):
    for name in files_in_dir:
        path = os.path.join(root, name)
        adapter_size += os.path.getsize(path)
        print(f"{name}: {os.path.getsize(path):,} bytes")
print(f"ИТОГО: {adapter_size:,} bytes ({adapter_size / 1024 / 1024:.2f} MB)")

In [ ]:
loss_history = [item for item in trainer.state.log_history if "loss" in item]
eval_history = [item for item in trainer.state.log_history if "eval_loss" in item]
steps = [item["step"] for item in loss_history]
losses = [item["loss"] for item in loss_history]
eval_steps = [item["step"] for item in eval_history]
eval_losses = [item["eval_loss"] for item in eval_history]

# На одном графике показываются train_loss и eval_loss по шагам обучения.
plt.figure(figsize=(8, 4))
plt.plot(steps, losses, marker="o", label="train_loss")
if eval_losses:
    plt.plot(eval_steps, eval_losses, marker="s", label="eval_loss")
plt.title("График loss во время QLoRA fine-tuning")
plt.xlabel("Шаг")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "loss_curve_variant20.png", dpi=150)
plt.show()

In [ ]:
fine_tuned_results = generate_text(trainer.model, test_prompts)
for item in fine_tuned_results:
    print("=" * 80)
    print("PROMPT:", item["prompt"])
    print("RESPONSE:", item["response"])

with open(RESULTS_DIR / "fine_tuned_outputs.json", "w", encoding="utf-8") as f:
    json.dump(fine_tuned_results, f, ensure_ascii=False, indent=2)

In [ ]:
comparison = []
for before, after in zip(base_results, fine_tuned_results):
    comparison.append(
        {
            "prompt": before["prompt"],
            "base_response": before["response"],
            "fine_tuned_response": after["response"],
        }
    )

with open(RESULTS_DIR / "comparison_variant20.json", "w", encoding="utf-8") as f:
    json.dump(comparison, f, ensure_ascii=False, indent=2)

# Сводка запуска собирает ключевые параметры и итоговые метрики в одном файле.
summary = {
    "variant": VARIANT,
    "topic": TOPIC,
    "dataset_size": len(normalized_dataset),
    "train_size": len(train_dataset),
    "validation_size": len(val_dataset),
    "model_name": MODEL_NAME,
    "max_seq_length": MAX_SEQ_LENGTH,
    "epochs": training_args.num_train_epochs,
    "batch_size": training_args.per_device_train_batch_size,
    "gradient_accumulation_steps": training_args.gradient_accumulation_steps,
    "learning_rate": training_args.learning_rate,
    "training_time_sec": training_time_sec,
    "validation_loss": eval_results.get("eval_loss"),
    "adapter_size_bytes": adapter_size,
    "lora_r": lora_config.r,
    "lora_alpha": lora_config.lora_alpha,
    "lora_dropout": lora_config.lora_dropout,
    "target_modules": sorted(list(lora_config.target_modules)),
}

with open(RESULTS_DIR / "run_summary_variant20.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

with open(RESULTS_DIR / "validation_metrics_variant20.json", "w", encoding="utf-8") as f:
    json.dump(eval_results, f, ensure_ascii=False, indent=2)

if USE_WANDB:
    wandb.finish()

print("Файлы результатов сохранены в", RESULTS_DIR)

## Дополнительно: сравнение рангов LoRA

Этот блок запускается отдельно от основного обучения.
По умолчанию сравнение отключено, чтобы базовый сценарий оставался быстрым.

Что делает блок:
- обучает несколько LoRA-адаптеров с разными значениями `r`;
- оценивает их на validation-наборе;
- сохраняет сводку сравнения и ответы моделей.

In [ ]:
RUN_RANK_COMPARISON = False
RANK_VALUES = [4, 16, 64]
RANK_COMPARISON_EPOCHS = 2
RANK_RESULTS_DIR = RESULTS_DIR / "rank_comparison"
RANK_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_RANK_COMPARISON =", RUN_RANK_COMPARISON)
print("RANK_VALUES =", RANK_VALUES)

In [ ]:
rank_comparison_results = []

if RUN_RANK_COMPARISON:
    for rank in RANK_VALUES:
        print("=" * 80)
        print(f"Запуск сравнения для rank={rank}")

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        rank_base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            quantization_config=bnb_config,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
        )
        rank_base_model.config.torch_dtype = torch.float16
        rank_base_model.config.use_cache = False

        rank_lora_config = LoraConfig(
            r=rank,
            lora_alpha=32,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
        )

        rank_model = prepare_model_for_kbit_training(rank_base_model)
        rank_model = get_peft_model(rank_model, rank_lora_config)

        rank_training_args = SFTConfig(
            output_dir=str(OUTPUT_DIR / f"rank_compare_r{rank}"),
            num_train_epochs=RANK_COMPARISON_EPOCHS,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            learning_rate=2e-4,
            warmup_steps=5,
            logging_steps=2,
            eval_strategy="epoch",
            save_strategy="no",
            optim="paged_adamw_8bit",
            fp16=False,
            bf16=False,
            per_device_eval_batch_size=1,
            report_to="none",
            lr_scheduler_type="cosine",
            gradient_checkpointing=True,
            max_grad_norm=0.3,
            dataset_text_field="text",
            max_length=MAX_SEQ_LENGTH,
            packing=False,
        )

        rank_trainer = SFTTrainer(
            model=rank_model,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            args=rank_training_args,
            processing_class=tokenizer,
        )

        rank_start_time = time.time()
        rank_trainer.train()
        rank_training_time_sec = time.time() - rank_start_time
        rank_eval_history = [item for item in rank_trainer.state.log_history if "eval_loss" in item]
        rank_eval_results = rank_eval_history[-1] if rank_eval_history else {}
        rank_outputs = generate_text(rank_trainer.model, test_prompts[:3], max_new_tokens=100)

        rank_adapter_dir = OUTPUT_DIR / f"rank_adapter_r{rank}"
        rank_trainer.model.save_pretrained(rank_adapter_dir)

        rank_adapter_size = 0
        for root, _, files_in_dir in os.walk(rank_adapter_dir):
            for name in files_in_dir:
                rank_adapter_size += os.path.getsize(os.path.join(root, name))

        rank_result = {
            "rank": rank,
            "epochs": RANK_COMPARISON_EPOCHS,
            "training_time_sec": rank_training_time_sec,
            "eval_loss": rank_eval_results.get("eval_loss"),
            "adapter_size_bytes": rank_adapter_size,
            "outputs": rank_outputs,
        }
        rank_comparison_results.append(rank_result)

        with open(RANK_RESULTS_DIR / f"rank_{rank}_summary.json", "w", encoding="utf-8") as f:
            json.dump(rank_result, f, ensure_ascii=False, indent=2)

        del rank_trainer
        del rank_model
        del rank_base_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    with open(RANK_RESULTS_DIR / "rank_comparison_summary.json", "w", encoding="utf-8") as f:
        json.dump(rank_comparison_results, f, ensure_ascii=False, indent=2)

    for item in rank_comparison_results:
        print(
            f"rank={item['rank']} | eval_loss={item['eval_loss']:.4f} | "
            f"time={item['training_time_sec']:.2f}s | adapter={item['adapter_size_bytes'] / 1024 / 1024:.2f} MB"
        )
else:
    print("Сравнение рангов отключено. Для запуска нужно установить RUN_RANK_COMPARISON = True.")

## Дополнительно: экспорт merged-модели в GGUF

Блок опциональный. Выполняется после основного обучения, когда LoRA-адаптер уже сохранен.
По умолчанию экспорт отключен.

In [ ]:
RUN_GGUF_EXPORT = False
MERGED_DIR = "/content/merged_variant20_qwen"
GGUF_F16_PATH = "/content/variant20-qwen-f16.gguf"
GGUF_Q4_PATH = "/content/variant20-qwen-q4_k_m.gguf"

print("RUN_GGUF_EXPORT =", RUN_GGUF_EXPORT)

In [ ]:
if RUN_GGUF_EXPORT:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Слияние LoRA-адаптера с базовой моделью перед экспортом в GGUF.
    merge_base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    merge_model = PeftModel.from_pretrained(merge_base_model, ADAPTER_DIR)
    merged_model = merge_model.merge_and_unload()
    merged_model.save_pretrained(MERGED_DIR)
    tokenizer.save_pretrained(MERGED_DIR)
    print("Merged model saved to", MERGED_DIR)
else:
    print("GGUF export disabled")

In [ ]:
if RUN_GGUF_EXPORT:
    !rm -rf /content/llama.cpp
    !git clone https://github.com/ggml-org/llama.cpp /content/llama.cpp
    !pip install -q -r /content/llama.cpp/requirements.txt
    !python /content/llama.cpp/convert_hf_to_gguf.py "$MERGED_DIR" --outfile "$GGUF_F16_PATH" --outtype f16
    print("GGUF f16 created:", GGUF_F16_PATH)
else:
    print("GGUF conversion skipped")

In [ ]:
if RUN_GGUF_EXPORT:
    !cmake -S /content/llama.cpp -B /content/llama.cpp/build
    !cmake --build /content/llama.cpp/build --config Release -j
    !/content/llama.cpp/build/bin/llama-quantize "$GGUF_F16_PATH" "$GGUF_Q4_PATH" q4_k_m
    print("GGUF q4_k_m created:", GGUF_Q4_PATH)
else:
    print("GGUF quantization skipped")

In [ ]:
if RUN_GGUF_EXPORT:
    gguf_summary = {
        "merged_dir": MERGED_DIR,
        "gguf_f16_path": GGUF_F16_PATH,
        "gguf_q4_path": GGUF_Q4_PATH,
        "gguf_f16_exists": os.path.exists(GGUF_F16_PATH),
        "gguf_q4_exists": os.path.exists(GGUF_Q4_PATH),
        "gguf_f16_size_bytes": os.path.getsize(GGUF_F16_PATH) if os.path.exists(GGUF_F16_PATH) else None,
        "gguf_q4_size_bytes": os.path.getsize(GGUF_Q4_PATH) if os.path.exists(GGUF_Q4_PATH) else None,
    }
    with open(RESULTS_DIR / "gguf_export_summary.json", "w", encoding="utf-8") as f:
        json.dump(gguf_summary, f, ensure_ascii=False, indent=2)
    print("GGUF export summary saved")
else:
    print("GGUF export summary skipped")

In [ ]:
!cd /content && zip -r -q variant20_qlora_results.zip variant20_qlora_output
print("Готов архив: /content/variant20_qlora_results.zip")

In [ ]:
from google.colab import files
files.download("/content/variant20_qlora_results.zip")

## Завершение

После выполнения ноутбука в папке результатов сохраняются артефакты обучения, сравнение генераций, метрики валидации и, при необходимости, дополнительные результаты сравнения рангов, WandB и GGUF.